# Text-to-text LLM Evaluation

## Import packages

In [ ]:
import nltk
import os
import string
import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from getpass import getpass
from unidecode import unidecode
from rouge_score import rouge_scorer
from sklearn.metrics.pairwise import cosine_similarity
from FactScoreLite import FactScore

from langchain_community.embeddings import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain_mistralai import ChatMistralAI
from langchain_ollama.llms import OllamaLLM as Ollama
from langchain.embeddings.cache import CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

## Disable warnings

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook:
1. MISTRAL_API_KEY
2. OPENAI_API_KEY
3. OPENAI_PROXY
4. TAVILY_API_KEY
5. ENTREZ_EMAIL

**Note:** FActScore metric requires an OpenAI API key to function properly, as it uses GPT models for fact extraction and verification.

## Import packages

In [ ]:
env_variables = [
  'MISTRAL_API_KEY',
  'OPENAI_API_KEY',
  'OPENAI_PROXY',
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Define evaluation function

In [ ]:
lemmatizer = nltk.stem.WordNetLemmatizer()

def preprocess(corpus: str) -> str:
  corpus = corpus.lower()
  stopset = nltk.corpus.stopwords.words('english') + nltk.corpus.stopwords.words('russian') + list(string.punctuation)
  tokens = nltk.word_tokenize(corpus)
  tokens = [t for t in tokens if t not in stopset]
  tokens = [lemmatizer.lemmatize(t) for t in tokens]
  corpus = ' '.join(tokens)
  corpus = unidecode(corpus)
  return corpus

In [ ]:
embeddings = OllamaEmbeddings(model='llama3.1')
store = LocalFileStore("./.embeddings_cache")

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
  embeddings,
  store,
  namespace=embeddings.model,
)

In [ ]:
def embeddings_cosine_sim_metric(expected_answers: list[str], predicted_answers: list[str]) -> float:
  results = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    expected_embedding = np.array(cached_embeddings.embed_query(expected_answer))
    predicted_embedding = np.array(cached_embeddings.embed_query(predicted_answer))

    sim = cosine_similarity(
      expected_embedding.reshape(1, -1),
      predicted_embedding.reshape(1, -1),
    )[0][0]

    results.append(sim)

  return np.mean(results)

In [ ]:
smoothie_f = nltk.translate.bleu_score.SmoothingFunction().method4

def bleu_metric(expected_answers, predicted_answers):
  scores = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    predicted_tokens = nltk.word_tokenize(predicted_answer)
    expected_tokens = [nltk.word_tokenize(expected_answer)]

    score = nltk.translate.bleu_score.sentence_bleu(
      expected_tokens,
      predicted_tokens,
      smoothing_function=smoothie_f,
    )

    scores.append(score)

  return np.mean(scores)

In [ ]:
rogue_l_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def rogue_l_metric(expected_answers, predicted_answers):
  scores = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    result = rogue_l_scorer.score(expected_answer, predicted_answer)

    scores.append(result['rougeL'])

  return np.mean(scores)

2025-07-06 03:18:15,113 - INFO - Using default tokenizer.


In [ ]:
rogue_1_scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)

def rogue_1_metric(expected_answers, predicted_answers):
  scores = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    result = rogue_1_scorer.score(expected_answer, predicted_answer)

    scores.append(result['rouge1'])

  return np.mean(scores)

2025-07-06 03:18:15,118 - INFO - Using default tokenizer.


In [ ]:
def factscore_metric(expected_answers, predicted_answers):
  try:
    fact_scorer = FactScore()
    scores = []

    for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
      try:
        score, _ = fact_scorer.get_factscore(
          generations=[predicted_answer],
          knowledge_sources=[expected_answer]
        )
        scores.append(score)
      except Exception as e:
        print(f"Error computing FActScore for pair: {e}")
        scores.append(0.0)

    return np.mean(scores) if scores else 0.0
  except Exception as e:
    print(f"Error initializing FActScore: {e}")
    return 0.0

In [ ]:
def eval_rag(chain) -> float:
  dataset_df = pd.read_csv('../datasets/mediqa.csv')
  expected_answers = dataset_df['answer']
  predicted_answers = []

  for index, row in tqdm(list(dataset_df.iterrows()), desc='Questions'):
    question = row['question']
    llm_answer = chain.invoke({'query': question})
    predicted_answers.append(llm_answer)

  cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
  bleu_score = bleu_metric(expected_answers, predicted_answers)
  rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
  rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)
  factscore = factscore_metric(expected_answers, predicted_answers)

  return cos_score, bleu_score, rogue_1_score, rogue_l_score, factscore

## Define prompt

In [ ]:
template = """
You are an assistant for question-answering tasks. Keep the answer verbose, with a minimum of three paragraphs.

QUERY: {query}

First, identify the key scientific concepts and data points that relate to the QUERY.
Then, analyze how these concepts connect to form a comprehensive answer.
Finally, synthesize your findings into a detailed response.
"""

prompt = PromptTemplate(
  template=template,
  input_variables=['query'],
)

## Setup LLMs

### Llama 3.1

In [ ]:
def get_llama3_1_llm(temperature=0):
  return Ollama(model='llama3.1', temperature=temperature)

### Mistral Large

In [ ]:
def get_mistral_large_llm(temperature=0):
  return ChatMistralAI(
    model='mistral-large-latest',
    temperature=temperature,
  )

### GPT-4o mini

In [ ]:
def get_chatgpt_40_mini_llm(temperature=0):
  return ChatOpenAI(
    model='gpt-4o-mini',
    temperature=temperature,
  )

### OpenBioLLM 70B Q2_k

In [ ]:
def get_openbiollm_70_Q2k_llm(temperature=0):
  return Ollama(model='taozhiyuai/openbiollm-llama-3:70b_q2_k', temperature=temperature)

### Biomistral 7B Q4_k_m

In [ ]:
def get_biomistral_q4_k_m_llm(temperature=0):
  return Ollama(model='cniongolo/biomistral', temperature=temperature)

## Evaluate the models

### Load QA dataset

In [ ]:
mediqa_df = pd.read_csv('../datasets/neurobiology_mediqa.csv')
mediqa_df

,question,answer
0,SSPE. My son is 33years of age and did not hav...,Subacute sclerosing panencephalitis: Subacute ...
1,Homozygout MTHFR A1298C Health Issues and long...,MTHFR gene variant (Inheritance): Because each...
2,What is Stroke?,Stroke: A stroke occurs when the blood supply ...
3,What causes Stroke?,Ischemic Stroke (Summary): Summary A stroke is...
4,What are the symptoms of Stroke?,What are the symptoms of Stroke?: The signs an...
5,What are the treatments of Stroke?,Stroke (Treatment): A stroke is a medical emer...
6,What is Dementia?,Dementia (WHAT IS DEMENTIA?): Dementia is the ...
7,What causes Dementia?,What causes Dementia?: Dementia usually occurs...
8,What are the symptoms of Dementia?,Dementia (Symptoms): Dementia symptoms include...
9,How to diagnose Dementia?,Dementia (Diagnosis): Diagnosing dementia and ...


### Setup experiment grid search parameters

In [ ]:
llms = (
  ('GPT-4o mini', get_chatgpt_40_mini_llm()),
  ('Mistral Large', get_mistral_large_llm()),
  ('LLaMA 3.1', get_llama3_1_llm()),
  ('OpenBioLLM 70B Q2_k', get_openbiollm_70_Q2k_llm()),
  ('Biomistral 7B Q4_k_m', get_biomistral_q4_k_m_llm()),
)

### Conduct the grid search

In [133]:
df = pd.DataFrame()

questions = mediqa_df['question'].tolist()
expected_answers = mediqa_df['answer'].tolist()

for llm_name, llm in llms:
  chain = prompt | llm | StrOutputParser()

  predicted_answers = []

  for question in tqdm(questions, desc='Questions'):
    response = chain.invoke(question)

    predicted_answers.append(response)

  # Evaluate metrics
  cos_sim = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
  bleu_score = bleu_metric(expected_answers, predicted_answers)
  rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
  rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)
  factscore = factscore_metric(expected_answers, predicted_answers)

  # Save results
  row = pd.DataFrame({
    'llm': llm_name,
    'cos_sim': cos_sim,
    'bleu': bleu_score,
    'rogue_1': rogue_1_score,
    'rogue_l': rogue_l_score,
    'factscore': factscore,
  }, index=[0])
  df = pd.concat([df, row], ignore_index=True)

df.sort_values(by='cos_sim', ascending=False)

Questions:   5%|▌         | 1/19 [00:11<03:18, 11.04s/it]


KeyboardInterrupt: 